In [1]:
!pip install requests beautifulsoup4 lxml openpyxl fake-useragent tqdm -q
print('Librerías instaladas correctamente')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 9.0 MB/s eta 0:00:00
Librerías instaladas correctamente


In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import re
from tqdm.notebook import tqdm
from IPython.display import display, HTML, clear_output
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings('ignore')

In [3]:
# ── Headers para simular un navegador real ──────────────────────────────────
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'es-CO,es;q=0.9,en;q=0.8',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}

In [4]:
# ── Configuración general ───────────────────────────────────────────────────
TIMEOUT       = 15    # segundos de espera por petición
DELAY         = 1.5   # segundos entre peticiones (respetar los servidores)
MAX_ARTICULOS = 20    # máximo de artículos por periódico (ajustable)

print(' Configuración cargada')
print(f'   → Timeout por petición: {TIMEOUT}s')
print(f'   → Delay entre peticiones: {DELAY}s')
print(f'   → Máximo artículos por periódico: {MAX_ARTICULOS}')

 Configuración cargada
   → Timeout por petición: 15s
   → Delay entre peticiones: 1.5s
   → Máximo artículos por periódico: 20


In [5]:
def fetch_page(url, timeout=TIMEOUT):
    """
    Descarga el HTML de una URL.
    Retorna un objeto BeautifulSoup o None si falla.
    """
    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'lxml')
        return soup
    except requests.exceptions.Timeout:
        print(f'    Timeout en: {url}')
        return None
    except requests.exceptions.HTTPError as e:
        print(f'    Error HTTP {e.response.status_code} en: {url}')
        return None
    except Exception as e:
        print(f'    Error inesperado en {url}: {str(e)[:60]}')
        return None


def limpiar_texto(texto):
    """
    Limpia un texto: elimina espacios múltiples, saltos de línea
    y caracteres de control.
    """
    if not texto:
        return ''
    texto = re.sub(r'[\n\r\t]+', ' ', str(texto))
    texto = re.sub(r' {2,}', ' ', texto)
    return texto.strip()


def mostrar_tabla_progreso(articulos_lista):
    """
    Muestra una tabla HTML en tiempo real con los últimos 5 artículos extraídos.
    """
    if not articulos_lista:
        return

    ultimos = articulos_lista[-5:]
    colores = {
        'La Silla Vacía': '#e8f4fd',
        'El Espectador':  '#e8fdf1',
        'El Tiempo':      '#fdf5e8'
    }
    filas_html = ''
    for a in ultimos:
        color   = colores.get(a.get('periodico', ''), '#ffffff')
        titular = a.get('titular', '')[:80] + ('...' if len(a.get('titular', '')) > 80 else '')
        filas_html += f"""
        <tr style='background-color:{color}'>
            <td style='padding:6px; border:1px solid #ddd; font-weight:bold'>{a.get('periodico','')}</td>
            <td style='padding:6px; border:1px solid #ddd'>{a.get('fecha_extraccion','')}</td>
            <td style='padding:6px; border:1px solid #ddd'>{titular}</td>
            <td style='padding:6px; border:1px solid #ddd; color:#666'>{len(a.get('texto_completo',''))} chars</td>
        </tr>"""

    html = f"""
    <div style='font-family:Arial; margin:10px 0'>
        <h4 style='color:#333'> Últimos artículos extraídos ({len(articulos_lista)} total)</h4>
        <table style='border-collapse:collapse; width:100%; font-size:13px'>
            <thead>
                <tr style='background:#2c3e50; color:white'>
                    <th style='padding:8px; border:1px solid #ddd'>Periódico</th>
                    <th style='padding:8px; border:1px solid #ddd'>Fecha extracción</th>
                    <th style='padding:8px; border:1px solid #ddd'>Titular</th>
                    <th style='padding:8px; border:1px solid #ddd'>Texto</th>
                </tr>
            </thead>
            <tbody>{filas_html}</tbody>
        </table>
    </div>"""
    clear_output(wait=True)
    display(HTML(html))


print(' Funciones auxiliares definidas')

 Funciones auxiliares definidas


In [14]:
from datetime import date

def scrape_silla_vacia(max_art=MAX_ARTICULOS):
    BASE_URL    = 'https://www.lasillavacia.com'
    articulos   = []
    urls_vistas = set()
    HOY         = date.today().strftime('%Y-%m-%d')   # ej: '2026-02-27'

    print(f'\n Iniciando scraping de LA SILLA VACÍA — filtrando por HOY ({HOY})...')

    # ── 1. Descargar portada ──────────────────────────────────────────────────
    soup_portada = fetch_page(BASE_URL)
    if not soup_portada:
        print('  ❌ No se pudo acceder a La Silla Vacía')
        return articulos

    # ── 2. DEBUG: ver qué links hay en la portada ─────────────────────────────
    # (ejecuta esto la primera vez para entender la estructura)
    todos_links = soup_portada.find_all('a', href=True)
    print(f'  🔍 Total de <a> en portada: {len(todos_links)}')
    # Mostrar los primeros 20 hrefs para diagnóstico
    hrefs_muestra = [l['href'] for l in todos_links if l['href'].startswith('/')][:20]
    print(f'  📋 Muestra de hrefs internos:')
    for h in hrefs_muestra:
        print(f'      {h}')

    # ── 3. Recolectar TODOS los links internos posibles ───────────────────────
    urls_candidatas = []
    for link in todos_links:
        href = link['href']

        # Descartar anclas, javascript, recursos estáticos y rutas de un solo nivel
        if (not href or href.startswith('#') or href.startswith('javascript')
                or '.' in href.split('/')[-1]   # archivos como .jpg, .css
                or href in ['/', '']):
            continue

        url_completa = href if href.startswith('http') else BASE_URL + href

        # Quedarnos solo con URLs del dominio lasillavacia.com
        if 'lasillavacia.com' not in url_completa:
            continue

        # Descartar secciones raíz (solo tienen 1 segmento después del dominio)
        path   = url_completa.replace(BASE_URL, '').strip('/')
        partes = [p for p in path.split('/') if p]
        if len(partes) < 2:
            continue

        # Descartar URLs de login, suscripción, etc.
        ignorar = ['login', 'registro', 'suscri', 'podcast', 'tag', 'autor',
                   'search', 'buscar', 'categoria', 'category', 'page', 'feed']
        if any(ig in path.lower() for ig in ignorar):
            continue

        if url_completa not in urls_vistas:
            urls_vistas.add(url_completa)
            urls_candidatas.append(url_completa)

    print(f'   URLs candidatas a artículos: {len(urls_candidatas)}')

    # ── 4. También revisar sección /historias y /silla-llena directamente ─────
    secciones_extra = [
        f'{BASE_URL}/historias',
        f'{BASE_URL}/silla-llena',
        f'{BASE_URL}/lasillallena',
        f'{BASE_URL}/noticias',
    ]
    for seccion_url in secciones_extra:
        soup_sec = fetch_page(seccion_url)
        if not soup_sec:
            continue
        time.sleep(DELAY)
        for link in soup_sec.find_all('a', href=True):
            href = link['href']
            url_completa = href if href.startswith('http') else BASE_URL + href
            if 'lasillavacia.com' not in url_completa or url_completa in urls_vistas:
                continue
            path   = url_completa.replace(BASE_URL, '').strip('/')
            partes = [p for p in path.split('/') if p]
            if len(partes) >= 2:
                urls_vistas.add(url_completa)
                urls_candidatas.append(url_completa)

    # Limitar para no tardar demasiado (revisaremos más de los necesarios
    # porque luego filtraremos por fecha)
    urls_candidatas = urls_candidatas[:max_art * 3]

    # ── 5. Scraping de cada artículo + filtro por fecha de hoy ───────────────
    print(f'  ⏳ Revisando {len(urls_candidatas)} URLs...')
    for url in tqdm(urls_candidatas, desc='La Silla Vacía', colour='blue'):
        if len(articulos) >= max_art:
            break

        time.sleep(DELAY)
        soup = fetch_page(url)
        if not soup:
            continue

        # ── Titular ──────────────────────────────────────────────────────────
        titular = ''
        # Primero intentar Open Graph (más confiable)
        meta_og = soup.find('meta', attrs={'property': 'og:title'})
        if meta_og:
            titular = limpiar_texto(meta_og.get('content', ''))
        # Fallback a h1
        if not titular:
            h1 = soup.find('h1')
            if h1:
                titular = limpiar_texto(h1.get_text())
        if not titular:
            continue

        # ── Fecha de publicación ──────────────────────────────────────────────
        fecha_publicacion = ''

        # Opción A: meta article:published_time  →  más confiable
        meta_fecha = soup.find('meta', attrs={'property': 'article:published_time'})
        if meta_fecha:
            fecha_publicacion = meta_fecha.get('content', '')

        # Opción B: etiqueta <time>
        if not fecha_publicacion:
            time_tag = soup.find('time')
            if time_tag:
                fecha_publicacion = (time_tag.get('datetime', '') or
                                     limpiar_texto(time_tag.get_text()))

        # Opción C: buscar patrón de fecha YYYY-MM-DD en la URL
        if not fecha_publicacion:
            match = re.search(r'(\d{4}[-/]\d{2}[-/]\d{2})', url)
            if match:
                fecha_publicacion = match.group(1)

        # Opción D: buscar en cualquier elemento con clase que contenga 'date' o 'fecha'
        if not fecha_publicacion:
            for selector in ['[class*="date"]', '[class*="fecha"]', '[class*="Date"]',
                              '[class*="time"]', '[class*="published"]']:
                elem = soup.select_one(selector)
                if elem:
                    texto_fecha = limpiar_texto(elem.get_text())
                    # Buscar patrón de fecha dentro del texto
                    match = re.search(r'(\d{4}[-/]\d{2}[-/]\d{2})', texto_fecha)
                    if match:
                        fecha_publicacion = match.group(1)
                        break

        # ── FILTRO: solo artículos de HOY ─────────────────────────────────────
        # Normalizamos la fecha encontrada para comparar con HOY (YYYY-MM-DD)
        fecha_normalizada = ''
        if fecha_publicacion:
            match = re.search(r'(\d{4})[-/](\d{2})[-/](\d{2})', fecha_publicacion)
            if match:
                fecha_normalizada = f'{match.group(1)}-{match.group(2)}-{match.group(3)}'

        # Si encontramos fecha y NO es hoy → saltar
        if fecha_normalizada and fecha_normalizada != HOY:
            continue

        # Si no encontramos fecha, incluimos el artículo (mejor no perder contenido)
        # pero lo marcamos como "fecha no detectada"
        if not fecha_normalizada:
            fecha_publicacion = 'No detectada — puede ser de hoy'

        # ── Texto del artículo ────────────────────────────────────────────────
        texto = ''
        for selector in ['div.article-content', 'div.entry-content',
                          'div[class*="content"]', 'div[class*="Content"]',
                          'div[class*="body"]', 'div[class*="Body"]',
                          'article', 'main']:
            elem = soup.select_one(selector)
            if elem:
                for tag in elem.find_all(['script', 'style', 'nav', 'aside',
                                          'footer', 'figure', 'iframe']):
                    tag.decompose()
                texto = limpiar_texto(elem.get_text())
                if len(texto) > 150:
                    break

        # Fallback: concatenar todos los párrafos
        if len(texto) < 150:
            texto = ' '.join([limpiar_texto(p.get_text())
                              for p in soup.find_all('p') if len(p.get_text()) > 40])

        # ── Autor ─────────────────────────────────────────────────────────────
        autor = ''
        meta_aut = soup.find('meta', attrs={'name': 'author'})
        if meta_aut:
            autor = meta_aut.get('content', '')
        if not autor:
            for selector in ['span[class*="author"]', 'a[rel="author"]',
                              'span[class*="Author"]', 'div[class*="author"]',
                              '[itemprop="author"]', '.byline']:
                elem = soup.select_one(selector)
                if elem:
                    autor = limpiar_texto(elem.get_text())
                    break

        # ── Descripción ───────────────────────────────────────────────────────
        descripcion = ''
        for attr in [{'property': 'og:description'}, {'name': 'description'}]:
            meta = soup.find('meta', attrs=attr)
            if meta:
                descripcion = meta.get('content', '')
                break

        # ── Sección ───────────────────────────────────────────────────────────
        seccion = ''
        meta_sec = soup.find('meta', attrs={'property': 'article:section'})
        if meta_sec:
            seccion = meta_sec.get('content', '')
        if not seccion:
            partes_url = url.replace(BASE_URL, '').strip('/').split('/')
            if len(partes_url) >= 2:
                seccion = partes_url[0].replace('-', ' ').title()

        # ── Etiquetas ─────────────────────────────────────────────────────────
        meta_tags = soup.find_all('meta', attrs={'property': 'article:tag'})
        etiquetas = ' | '.join([t.get('content', '') for t in meta_tags])

        # ── Imagen ────────────────────────────────────────────────────────────
        imagen_principal = ''
        meta_img = soup.find('meta', attrs={'property': 'og:image'})
        if meta_img:
            imagen_principal = meta_img.get('content', '')

        articulos.append({
            'periodico':         'La Silla Vacía',
            'url':               url,
            'titular':           titular,
            'descripcion':       descripcion,
            'texto_completo':    texto[:5000],
            'autor':             autor,
            'fecha_publicacion': fecha_publicacion,
            'fecha_extraccion':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'seccion':           seccion,
            'etiquetas':         etiquetas,
            'imagen_principal':  imagen_principal,
            'num_palabras':      len(texto.split()),
            'num_caracteres':    len(texto),
        })

    print(f'   La Silla Vacía: {len(articulos)} artículos de HOY ({HOY}) extraídos')
    return articulos

In [7]:
def scrape_el_espectador(max_art=MAX_ARTICULOS):
    BASE_URL    = 'https://www.elespectador.com'
    articulos   = []
    urls_vistas = set()

    print('\n Iniciando scraping de EL ESPECTADOR...')

    soup_portada = fetch_page(BASE_URL)
    if not soup_portada:
        print('   No se pudo acceder a El Espectador')
        return articulos

    links_raw      = soup_portada.find_all('a', href=True)
    urls_articulos = []
    SECCIONES      = ['/colombia/', '/mundo/', '/opinion/', '/economia/',
                      '/cultura/', '/judicial/', '/politica/', '/tecnologia/',
                      '/ciencia/', '/salud/', '/ambiente/']

    for link in links_raw:
        href = link['href']
        if any(s in href for s in SECCIONES):
            url_completa = href if href.startswith('http') else BASE_URL + href
            if url_completa not in urls_vistas and 'elespectador.com' in url_completa:
                partes = url_completa.replace(BASE_URL, '').strip('/').split('/')
                if len(partes) >= 2:
                    urls_vistas.add(url_completa)
                    urls_articulos.append(url_completa)

    urls_articulos = list(dict.fromkeys(urls_articulos))[:max_art]
    print(f'   URLs encontradas: {len(urls_articulos)}')

    for url in tqdm(urls_articulos, desc='El Espectador', colour='green'):
        time.sleep(DELAY)
        soup = fetch_page(url)
        if not soup:
            continue

        # Titular
        titular = ''
        for selector in ['h1.article-title', 'h1.Title', 'h1[class*="Title"]',
                          'h1[class*="title"]', 'h1[class*="headline"]', 'h1']:
            elem = soup.select_one(selector)
            if elem:
                titular = limpiar_texto(elem.get_text())
                break
        if not titular:
            meta = soup.find('meta', attrs={'property': 'og:title'})
            if meta:
                titular = meta.get('content', '')
        if not titular:
            continue

        # Texto
        texto = ''
        for selector in ['div.article-content', 'div[class*="ArticleBody"]',
                          'div[class*="article-body"]', 'div[class*="Content"]', 'article', 'main']:
            elem = soup.select_one(selector)
            if elem:
                for tag in elem.find_all(['script', 'style', 'nav', 'aside', 'footer', 'figure']):
                    tag.decompose()
                texto = limpiar_texto(elem.get_text())
                if len(texto) > 100:
                    break

        # Autor
        autor = ''
        for selector in ['span[class*="author"]', 'a[class*="author"]', 'p[class*="author"]',
                          'div[class*="author"]', '[itemprop="author"]']:
            elem = soup.select_one(selector)
            if elem:
                autor = limpiar_texto(elem.get_text())
                break

        # Fecha
        fecha_publicacion = ''
        for selector in ['time', '[class*="date"]', '[class*="Date"]',
                          'meta[property="article:published_time"]']:
            elem = soup.select_one(selector)
            if elem:
                fecha_publicacion = (elem.get('datetime', '') or
                                     elem.get('content', '') or
                                     limpiar_texto(elem.get_text()))
                if fecha_publicacion:
                    break

        # Descripción
        descripcion = ''
        for attr in [{'name': 'description'}, {'property': 'og:description'}]:
            meta = soup.find('meta', attrs=attr)
            if meta:
                descripcion = meta.get('content', '')
                break

        # Sección (de meta o inferida de URL)
        seccion = ''
        meta_sec = soup.find('meta', attrs={'property': 'article:section'})
        if meta_sec:
            seccion = meta_sec.get('content', '')
        if not seccion:
            partes_url = url.replace(BASE_URL, '').strip('/').split('/')
            if partes_url:
                seccion = partes_url[0].replace('-', ' ').title()

        # Etiquetas
        meta_tags = soup.find_all('meta', attrs={'property': 'article:tag'})
        etiquetas = ' | '.join([t.get('content', '') for t in meta_tags])

        # Imagen principal
        imagen_principal = ''
        meta_img = soup.find('meta', attrs={'property': 'og:image'})
        if meta_img:
            imagen_principal = meta_img.get('content', '')

        articulos.append({
            'periodico':         'El Espectador',
            'url':               url,
            'titular':           titular,
            'descripcion':       descripcion,
            'texto_completo':    texto[:5000],
            'autor':             autor,
            'fecha_publicacion': fecha_publicacion,
            'fecha_extraccion':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'seccion':           seccion,
            'etiquetas':         etiquetas,
            'imagen_principal':  imagen_principal,
            'num_palabras':      len(texto.split()),
            'num_caracteres':    len(texto),
        })

    print(f'   El Espectador: {len(articulos)} artículos extraídos')
    return articulos

print(' Función scrape_el_espectador() definida')

 Función scrape_el_espectador() definida


In [9]:
def scrape_el_tiempo(max_art=MAX_ARTICULOS):
    BASE_URL    = 'https://www.eltiempo.com'
    articulos   = []
    urls_vistas = set()

    print('\n Iniciando scraping de EL TIEMPO...')

    paginas_entrada = [
        BASE_URL,
        f'{BASE_URL}/colombia',
        f'{BASE_URL}/mundo',
        f'{BASE_URL}/economia',
        f'{BASE_URL}/vida',
    ]

    urls_articulos = []
    for pagina in paginas_entrada:
        soup_portada = fetch_page(pagina)
        if not soup_portada:
            continue
        time.sleep(DELAY)

        for link in soup_portada.find_all('a', href=True):
            href         = link['href']
            url_completa = href if href.startswith('http') else BASE_URL + href

            if 'eltiempo.com' not in url_completa or url_completa in urls_vistas:
                continue

            path   = url_completa.replace(BASE_URL, '').strip('/')
            partes = path.split('/')
            if len(partes) >= 2 and any(c.isdigit() for c in partes[-1]):
                urls_vistas.add(url_completa)
                urls_articulos.append(url_completa)
            elif len(partes) >= 3:
                urls_vistas.add(url_completa)
                urls_articulos.append(url_completa)

    urls_articulos = urls_articulos[:max_art]
    print(f'   URLs encontradas: {len(urls_articulos)}')

    for url in tqdm(urls_articulos, desc='El Tiempo', colour='#FF8C00'):
        time.sleep(DELAY)
        soup = fetch_page(url)
        if not soup:
            continue

        # Titular (primero og:title, luego h1)
        titular  = ''
        meta_og  = soup.find('meta', attrs={'property': 'og:title'})
        if meta_og:
            titular = meta_og.get('content', '')
        if not titular:
            for selector in ['h1[class*="title"]', 'h1[class*="Title"]',
                              'h1[class*="headline"]', 'h1']:
                elem = soup.select_one(selector)
                if elem:
                    titular = limpiar_texto(elem.get_text())
                    break
        if not titular:
            continue

        # Texto
        texto = ''
        for selector in ['div[class*="article-content"]', 'div[class*="ArticleBody"]',
                          'div[class*="story-body"]', 'article',
                          'section[class*="content"]', 'main']:
            elem = soup.select_one(selector)
            if elem:
                for tag in elem.find_all(['script', 'style', 'nav', 'aside',
                                          'footer', 'figure', 'iframe']):
                    tag.decompose()
                texto = limpiar_texto(elem.get_text())
                if len(texto) > 100:
                    break
        if len(texto) < 100:   # fallback: todos los párrafos
            texto = ' '.join([limpiar_texto(p.get_text())
                              for p in soup.find_all('p') if len(p.get_text()) > 30])

        # Autor
        autor     = ''
        meta_aut  = soup.find('meta', attrs={'name': 'author'})
        if meta_aut:
            autor = meta_aut.get('content', '')
        if not autor:
            for selector in ['[class*="author"]', '[class*="Author"]', '[itemprop="author"]']:
                elem = soup.select_one(selector)
                if elem:
                    autor = limpiar_texto(elem.get_text())
                    break

        # Fecha
        fecha_publicacion = ''
        meta_fecha        = soup.find('meta', attrs={'property': 'article:published_time'})
        if meta_fecha:
            fecha_publicacion = meta_fecha.get('content', '')
        if not fecha_publicacion:
            for selector in ['time', '[class*="date"]', '[class*="Date"]']:
                elem = soup.select_one(selector)
                if elem:
                    fecha_publicacion = (elem.get('datetime', '') or
                                         limpiar_texto(elem.get_text()))
                    if fecha_publicacion:
                        break

        # Descripción
        descripcion = ''
        for attr in [{'name': 'description'}, {'property': 'og:description'}]:
            meta = soup.find('meta', attrs=attr)
            if meta:
                descripcion = meta.get('content', '')
                break

        # Sección
        seccion  = ''
        meta_sec = soup.find('meta', attrs={'property': 'article:section'})
        if meta_sec:
            seccion = meta_sec.get('content', '')
        if not seccion:
            partes_url = url.replace(BASE_URL, '').strip('/').split('/')
            if partes_url:
                seccion = partes_url[0].replace('-', ' ').title()

        # Etiquetas
        meta_tags = soup.find_all('meta', attrs={'property': 'article:tag'})
        etiquetas = ' | '.join([t.get('content', '') for t in meta_tags])

        # Imagen principal
        imagen_principal = ''
        meta_img         = soup.find('meta', attrs={'property': 'og:image'})
        if meta_img:
            imagen_principal = meta_img.get('content', '')

        articulos.append({
            'periodico':         'El Tiempo',
            'url':               url,
            'titular':           titular,
            'descripcion':       descripcion,
            'texto_completo':    texto[:5000],
            'autor':             autor,
            'fecha_publicacion': fecha_publicacion,
            'fecha_extraccion':  datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'seccion':           seccion,
            'etiquetas':         etiquetas,
            'imagen_principal':  imagen_principal,
            'num_palabras':      len(texto.split()),
            'num_caracteres':    len(texto),
        })

    print(f'   El Tiempo: {len(articulos)} artículos extraídos')
    return articulos

print(' Función scrape_el_tiempo() definida')

 Función scrape_el_tiempo() definida


In [15]:
todos_los_articulos = []

inicio = datetime.now()
print(f' Iniciando scraping a las {inicio.strftime("%H:%M:%S")}\n')
print('=' * 60)

art_silla = scrape_silla_vacia(max_art=MAX_ARTICULOS)
todos_los_articulos.extend(art_silla)
mostrar_tabla_progreso(todos_los_articulos)

art_espectador = scrape_el_espectador(max_art=MAX_ARTICULOS)
todos_los_articulos.extend(art_espectador)
mostrar_tabla_progreso(todos_los_articulos)

art_tiempo = scrape_el_tiempo(max_art=MAX_ARTICULOS)
todos_los_articulos.extend(art_tiempo)
mostrar_tabla_progreso(todos_los_articulos)

fin      = datetime.now()
duracion = (fin - inicio).seconds

print('\n' + '=' * 60)
print(f'SCRAPING COMPLETADO en {duracion} segundos')
print(f'    La Silla Vacía:  {len(art_silla)} artículos')
print(f'    El Espectador:   {len(art_espectador)} artículos')
print(f'    El Tiempo:       {len(art_tiempo)} artículos')
print(f'    TOTAL:           {len(todos_los_articulos)} artículos')

Periódico,Fecha extracción,Titular,Texto
El Tiempo,2026-02-27 10:23:40,‘El Gobierno Nacional no puede insistir en implementar la reforma a la salud y l...,5000 chars
El Tiempo,2026-02-27 10:23:42,"Gobierno trasladará a millones de usuarios a otras EPS, pese a crisis en varias ...",5000 chars
El Tiempo,2026-02-27 10:23:43,Los 22 contratos de familiares de ministro Guillermo Jaramillo que fueron adjudi...,5000 chars
El Tiempo,2026-02-27 10:23:45,Se mueve caso de niñas envenenadas con talio en frambuesas: en 72 horas hubo tre...,5000 chars
El Tiempo,2026-02-27 10:23:46,Vuelve la amenaza de la 'expropiación exprés' a través del nuevo decreto de emer...,5000 chars



SCRAPING COMPLETADO en 149 segundos
    La Silla Vacía:  7 artículos
    El Espectador:   7 artículos
    El Tiempo:       19 artículos
    TOTAL:           33 artículos


In [16]:
df = pd.DataFrame(todos_los_articulos)

columnas_orden = [
    'periodico', 'fecha_publicacion', 'fecha_extraccion', 'seccion',
    'titular', 'descripcion', 'texto_completo', 'autor',
    'etiquetas', 'num_palabras', 'num_caracteres', 'url', 'imagen_principal'
]
columnas_presentes = [c for c in columnas_orden if c in df.columns]
df = df[columnas_presentes]

print(f'📊 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas\n')

resumen = df.groupby('periodico').agg(
    articulos        = ('titular', 'count'),
    palabras_promedio = ('num_palabras', 'mean'),
    con_autor        = ('autor',  lambda x: (x != '').sum()),
    con_fecha        = ('fecha_publicacion', lambda x: (x != '').sum())
).round(0)

display(resumen)
display(df[['periodico', 'titular', 'autor', 'seccion', 'num_palabras']].head())

📊 Dimensiones: 33 filas × 13 columnas



,articulos,palabras_promedio,con_autor,con_fecha
periodico,,,,
El Espectador,7,26.0,0,6
El Tiempo,19,914.0,17,14
La Silla Vacía,7,521.0,5,7


,periodico,titular,autor,seccion,num_palabras
0,La Silla Vacía,Tarjetón de candidatos al Congreso 2026 - 2030,La Silla Vacía,Https:,47
1,La Silla Vacía,Programa de Inmersión Colombia 2026,,Programa De Inmersion Colombia 2026,783
2,La Silla Vacía,Curso virtual: ¿Qué está pasando en Colombia? ...,,Curso Virtual Que Esta Pasando En Colombia 2026,956
3,La Silla Vacía,"Roy pide votar por hermano de Eduardo Pulgar, ...",La Silla Vacía,En Vivo,389
4,La Silla Vacía,Duerma informado con las movidas de este 26 de...,La Silla Vacía,En Vivo,479


In [17]:
def crear_excel_profesional(df, nombre_archivo='periodicos_colombia.xlsx'):
    wb = openpyxl.Workbook()

    COLORES = {
        'La Silla Vacía': {'fila': 'DBEAFE', 'header': '1D4ED8'},
        'El Espectador':  {'fila': 'DCFCE7', 'header': '15803D'},
        'El Tiempo':      {'fila': 'FEF3C7', 'header': 'B45309'},
    }
    HEADER_DEFAULT = '374151'

    # ── Hoja 1: datos completos ────────────────────────────────────────────
    ws1 = wb.active
    ws1.title = 'Artículos'

    nombres_col = {
        'periodico': 'Periódico', 'fecha_publicacion': 'Fecha Publicación',
        'fecha_extraccion': 'Fecha Extracción', 'seccion': 'Sección',
        'titular': 'Titular', 'descripcion': 'Descripción (Lead)',
        'texto_completo': 'Texto Completo', 'autor': 'Autor/a',
        'etiquetas': 'Etiquetas', 'num_palabras': 'N° Palabras',
        'num_caracteres': 'N° Caracteres', 'url': 'URL',
        'imagen_principal': 'Imagen Principal (URL)',
    }

    columnas = list(df.columns)
    headers  = [nombres_col.get(c, c.replace('_', ' ').title()) for c in columnas]

    for ci, h in enumerate(headers, 1):
        cell            = ws1.cell(row=1, column=ci, value=h)
        cell.font       = Font(bold=True, color='FFFFFF', name='Arial', size=10)
        cell.fill       = PatternFill('solid', start_color=HEADER_DEFAULT)
        cell.alignment  = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border     = Border(bottom=Side(style='medium', color='000000'),
                                  right=Side(style='thin', color='CCCCCC'))
    ws1.row_dimensions[1].height = 30

    for ri, row in enumerate(df.itertuples(index=False), 2):
        periodico  = getattr(row, 'periodico', '')
        color_fila = COLORES.get(periodico, {}).get('fila', 'FFFFFF')

        for ci, col_name in enumerate(columnas, 1):
            valor = getattr(row, col_name, '')
            if pd.isna(valor):
                valor = ''
            cell           = ws1.cell(row=ri, column=ci, value=str(valor) if valor else '')
            cell.font      = Font(name='Arial', size=9)
            cell.fill      = PatternFill('solid', start_color=color_fila)
            cell.alignment = Alignment(
                vertical='top',
                wrap_text=(col_name in ['titular', 'texto_completo', 'descripcion'])
            )
            cell.border = Border(bottom=Side(style='thin', color='E5E7EB'))

        ws1.row_dimensions[ri].height = 60

    anchos = {
        'periodico': 18, 'fecha_publicacion': 20, 'fecha_extraccion': 20,
        'seccion': 18, 'titular': 55, 'descripcion': 50, 'texto_completo': 80,
        'autor': 22, 'etiquetas': 35, 'num_palabras': 14, 'num_caracteres': 16,
        'url': 50, 'imagen_principal': 50,
    }
    for ci, col_name in enumerate(columnas, 1):
        ws1.column_dimensions[get_column_letter(ci)].width = anchos.get(col_name, 20)

    ws1.auto_filter.ref = ws1.dimensions
    ws1.freeze_panes    = 'A2'

    # ── Hoja 2: resumen ────────────────────────────────────────────────────
    ws2 = wb.create_sheet('Resumen')

    ws2.merge_cells('A1:F1')
    t        = ws2['A1']
    t.value  = '📊 Resumen del Scraping de Periódicos Colombianos'
    t.font   = Font(bold=True, size=14, name='Arial', color='1E293B')
    t.alignment = Alignment(horizontal='center', vertical='center')
    t.fill   = PatternFill('solid', start_color='F1F5F9')
    ws2.row_dimensions[1].height = 35

    ws2['A2'] = f'Extracción: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}'
    ws2['A2'].font = Font(italic=True, color='6B7280', size=9)

    heads_res = ['Periódico', 'Total Artículos', 'Con Titular',
                  'Con Autor', 'Con Fecha', 'Promedio Palabras']
    for ci, h in enumerate(heads_res, 1):
        cell           = ws2.cell(row=4, column=ci, value=h)
        cell.font      = Font(bold=True, color='FFFFFF', name='Arial', size=10)
        cell.fill      = PatternFill('solid', start_color='1E293B')
        cell.alignment = Alignment(horizontal='center', vertical='center')
    ws2.row_dimensions[4].height = 25

    for ri, per in enumerate(df['periodico'].unique(), 5):
        sub        = df[df['periodico'] == per]
        color_fila = COLORES.get(per, {}).get('fila', 'FFFFFF')
        datos      = [per, len(sub),
                      (sub['titular'] != '').sum(),
                      (sub['autor'] != '').sum(),
                      (sub['fecha_publicacion'] != '').sum(),
                      round(sub['num_palabras'].mean(), 0)]
        for ci, val in enumerate(datos, 1):
            cell           = ws2.cell(row=ri, column=ci, value=val)
            cell.font      = Font(name='Arial', size=10)
            cell.fill      = PatternFill('solid', start_color=color_fila)
            cell.alignment = Alignment(horizontal='center' if ci > 1 else 'left',
                                        vertical='center')
        ws2.row_dimensions[ri].height = 22

    fila_total = len(df['periodico'].unique()) + 5
    totales    = ['TOTAL', len(df),
                  (df['titular'] != '').sum(),
                  (df['autor'] != '').sum(),
                  (df['fecha_publicacion'] != '').sum(),
                  round(df['num_palabras'].mean(), 0)]
    for ci, val in enumerate(totales, 1):
        cell           = ws2.cell(row=fila_total, column=ci, value=val)
        cell.font      = Font(bold=True, name='Arial', size=10)
        cell.fill      = PatternFill('solid', start_color='E2E8F0')
        cell.alignment = Alignment(horizontal='center' if ci > 1 else 'left',
                                    vertical='center')

    for ci, w in enumerate([25, 18, 14, 14, 14, 20], 1):
        ws2.column_dimensions[get_column_letter(ci)].width = w

    wb.save(nombre_archivo)
    return nombre_archivo


# ── Ejecutar ─────────────────────────────────────────────────────────────────
nombre_archivo = f'periodicos_colombia_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'

if len(df) > 0:
    ruta = crear_excel_profesional(df, nombre_archivo)
    print(f' Excel generado: {ruta}')
    print(f'    Artículos totales: {len(df)}')
    print(f'   Hojas: Artículos + Resumen')
else:
    print('  No hay artículos. Revisa la conexión o los scrapers.')

 Excel generado: periodicos_colombia_20260227_1023.xlsx
    Artículos totales: 33
   Hojas: Artículos + Resumen
